# FastF1 → GCS cache seeder (Colab / local)

**Purpose:** download FastF1 livetiming on a host with a *working IP* (Colab or your laptop), then upload the cache to GCS so Cloud Run Jobs can import **without** hitting livetiming.

Cloud Run and GCE soft-fail livetiming (`Failed to load timing data` → `DataNotLoadedError`). This notebook is the supported seed path.

| Env | Default |
|-----|---------|
| Bucket prefix | `gs://f1-data-library-fastf1-cache/fastf1` |
| Local cache | `./fastf1-cache` (Colab) or `~/.cache/fastf1` (local) |

### How to run
1. Open in **Colab** (recommended) or Jupyter locally.
2. Edit **Config** cell (year / race / sessions).
3. Authenticate to GCP (cell below).
4. Run **Install → Seed → Upload → Verify**.
5. Retry the Cloud Run job (`INGEST_SOURCE=fastf1`) or leave OpenF1 as default and use this only for FastF1 enrichment.


## 1) Install dependencies


In [ ]:
%pip install -q fastf1==3.8.3 google-cloud-storage>=2.14.0 pandas numpy
print("Installed fastf1 + google-cloud-storage")


## 2) Config — edit these


In [ ]:
from pathlib import Path
import os
import sys

# ── Edit me ──────────────────────────────────────────────────────────────────
YEAR = 2026
RACE = "Dutch Grand Prix"          # e.g. "Monaco Grand Prix", "British Grand Prix"
SESSIONS = ["FP1"]                # Dutch 2026 is sprint: FP1, SQ, S, Q, R (no FP2/FP3)
GCS_URI = "gs://f1-data-library-fastf1-cache/fastf1"
SKIP_UPLOAD = False                  # True = load only, no GCS push
# ─────────────────────────────────────────────────────────────────────────────

# Colab uses a project-local cache; laptop prefers the shared FastF1 home cache.
IN_COLAB = "google.colab" in sys.modules
CACHE_DIR = Path("/content/fastf1-cache") if IN_COLAB else Path.home() / ".cache" / "fastf1"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

os.environ["FASTF1_CACHE_DIR"] = str(CACHE_DIR)
os.environ["FASTF1_CACHE_GCS_URI"] = GCS_URI

print(f"year={YEAR} race={RACE!r} sessions={SESSIONS}")
print(f"cache_dir={CACHE_DIR}")
print(f"gcs_uri={GCS_URI}")
print(f"in_colab={IN_COLAB} skip_upload={SKIP_UPLOAD}")


## 3) Authenticate to Google Cloud

**Colab:** run the next cell and pick the Google account that can write to `f1-data-library`.

**Local:** skip that cell and use Application Default Credentials instead:

```bash
gcloud auth application-default login
gcloud config set project f1-data-library
```

The identity needs `roles/storage.objectAdmin` (or equivalent) on `gs://f1-data-library-fastf1-cache`.


In [ ]:
# Colab-only auth. Safe no-op elsewhere.
try:
    from google.colab import auth as colab_auth
    colab_auth.authenticate_user()
    print("Colab user authenticated")
except ImportError:
    print("Not Colab — using Application Default Credentials / GOOGLE_APPLICATION_CREDENTIALS")


## 4) Helpers (GCS push + cache inventory)

Self-contained copies of the worker helpers so this notebook runs without cloning the repo.


In [ ]:
from __future__ import annotations

from pathlib import Path
from urllib.parse import urlparse


def parse_gcs_uri(uri: str) -> tuple[str, str]:
    raw = uri.strip()
    if not raw.startswith("gs://"):
        raise ValueError(f"GCS URI must start with gs:// (got {uri!r})")
    parsed = urlparse(raw)
    bucket = parsed.netloc
    prefix = parsed.path.lstrip("/")
    if prefix and not prefix.endswith("/"):
        prefix += "/"
    if not bucket:
        raise ValueError(f"Invalid GCS URI: {uri!r}")
    return bucket, prefix


def push_cache(local_dir: Path, uri: str) -> dict:
    from google.cloud import storage

    bucket_name, prefix = parse_gcs_uri(uri)
    client = storage.Client()
    bucket = client.bucket(bucket_name)

    pushed = skipped = 0
    for path in local_dir.rglob("*"):
        if not path.is_file():
            continue
        rel = path.relative_to(local_dir).as_posix()
        blob = bucket.blob(f"{prefix}{rel}")
        if blob.exists():
            blob.reload()
            if blob.size == path.stat().st_size:
                skipped += 1
                continue
        blob.upload_from_filename(str(path))
        pushed += 1
        print(f"  ↑ {rel}")

    print(f"GCS push complete: uploaded={pushed} unchanged={skipped} → {uri}")
    return {"pushed": pushed, "skipped": skipped, "uri": uri}


def list_local_sessions(cache_dir: Path, year: int, race: str) -> list[dict]:
    root = cache_dir / str(year)
    if not root.exists():
        return []
    token = race.replace(" ", "_")
    out = []
    for event_dir in sorted(p for p in root.iterdir() if p.is_dir() and token in p.name):
        for session_dir in sorted(p for p in event_dir.iterdir() if p.is_dir()):
            files = [f.name for f in session_dir.iterdir() if f.is_file()]
            out.append({
                "event": event_dir.name,
                "session": session_dir.name,
                "file_count": len(files),
                "has_timing": "timing_app_data.ff1pkl" in files,
            })
    return out


def list_gcs_events(uri: str, year: int) -> list[str]:
    from google.cloud import storage

    bucket_name, prefix = parse_gcs_uri(uri)
    client = storage.Client()
    year_prefix = f"{prefix}{year}/"
    events = set()
    for blob in client.list_blobs(bucket_name, prefix=year_prefix):
        rel = blob.name[len(year_prefix):]
        if not rel:
            continue
        events.add(rel.split("/", 1)[0])
    return sorted(events)

print("Helpers ready")


## 5) Seed FastF1 sessions into local cache

This is the step that **must** run on a non-datacenter IP. If you see `Failed to load timing data` / `DataNotLoadedError`, stop — Colab/laptop egress is blocked for this network too; try another network or stick to OpenF1.


In [ ]:
import fastf1

fastf1.Cache.enable_cache(str(CACHE_DIR))
print(f"FastF1 cache enabled: {CACHE_DIR}")

results = []
for stype in SESSIONS:
    print(f"\n── Loading {YEAR} {RACE} {stype} ──")
    session = fastf1.get_session(YEAR, RACE, stype)
    session.load(telemetry=True, laps=True, weather=False)
    n_drivers = len(session.drivers)
    n_laps = len(session.laps)
    print(f"✅ {stype}: {n_drivers} drivers, {n_laps} laps")
    results.append({"session": stype, "drivers": n_drivers, "laps": n_laps})

print("\nSeed summary:")
for row in results:
    print(f"  {row}")

local_inv = list_local_sessions(CACHE_DIR, YEAR, RACE)
print(f"\nLocal cache folders for {YEAR} {RACE!r}:")
for row in local_inv:
    flag = "timing=yes" if row["has_timing"] else "timing=NO"
    print(f"  {row['event']}/{row['session']}: {row['file_count']} files ({flag})")

if not local_inv:
    raise RuntimeError("No local session folders found after load — seed failed")
if any(not r["has_timing"] for r in local_inv):
    print("⚠️  Some folders lack timing_app_data.ff1pkl — Cloud Run may still fail for those sessions")


## 6) Upload cache to GCS


In [ ]:
if SKIP_UPLOAD:
    print("SKIP_UPLOAD=True — not pushing to GCS")
else:
    push_result = push_cache(CACHE_DIR, GCS_URI)
    print(push_result)


## 7) Verify remote inventory

Confirm the race now appears under the year prefix in GCS (same check the worker uses before fail-fast).


In [ ]:
events = list_gcs_events(GCS_URI, YEAR)
token = RACE.replace(" ", "_")
matching = [e for e in events if token in e]

print(f"GCS {YEAR} events ({len(events)}):")
for e in events:
    mark = " ← MATCH" if token in e else ""
    print(f"  {e}{mark}")

if not matching:
    raise RuntimeError(
        f"Race token {token!r} not found under {GCS_URI}/{YEAR}/ — upload incomplete"
    )

print(f"\n✅ Seed OK — Cloud Run can pull cache for: {matching}")
print("Next: retry the Jobs Admin import with INGEST_SOURCE=fastf1,")
print("or keep OpenF1 default and use this cache only when you need FastF1 fidelity.")


## Troubleshooting

| Symptom | Likely cause | What to do |
|---------|--------------|------------|
| `Failed to load timing data` / `DataNotLoadedError` | Datacenter / blocked IP | Use Colab on a consumer network, or run locally at home |
| GCS `403` / `PermissionDenied` | Wrong Google account | Re-auth; need objectAdmin on the bucket |
| Verify finds Miami/Monaco but not your race | Upload skipped or wrong `RACE` string | Re-run seed + upload; match FastF1 event name exactly |
| Cloud Run still fail-fast | Cache prefix mismatch | Worker expects `FASTF1_CACHE_GCS_URI=gs://f1-data-library-fastf1-cache/fastf1` |

Companion CLI (from `front-end` branch):

```bash
export FASTF1_CACHE_GCS_URI=gs://f1-data-library-fastf1-cache/fastf1
python3 worker/pipeline/seed_fastf1_cache.py \
  --year 2026 --race "British Grand Prix" --session Q R
```
